In [1]:
import os
from PIL import Image

DATASET_DIR = "dataset"

def fix_image(path):
    try:
        img = Image.open(path)

        # Convert images with palette/alpha → JPG
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")

        # Convert webp/png to jpg
        if path.lower().endswith((".webp", ".png")):
            new_path = path.rsplit(".", 1)[0] + ".jpg"
            img.save(new_path, "JPEG")
            os.remove(path)
            print(f"Converted: {path} → {new_path}")

        else:
            # Just verify JPG
            img.verify()

    except Exception as e:
        print("Corrupted or unreadable → Removing:", path)
        os.remove(path)

# Run on all folders
for folder in ["tomato", "Radish", "Carrot", "Potato"]:
    subdir = os.path.join(DATASET_DIR, folder)
    for file in os.listdir(subdir):
        fix_image(os.path.join(subdir, file))

print("Dataset cleaned successfully!")


Dataset cleaned successfully!


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np

# -------------------------
# 1. DATASET PATH
# -------------------------
dataset_path = "dataset"   # your folder structure

# -------------------------
# 2. IMAGE DATA GENERATOR
# -------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

num_classes = len(train_data.class_indices)
print("Detected Classes:", train_data.class_indices)

# -------------------------
# 3. LOAD BASE MODEL (TRANSFER LEARNING)
# -------------------------
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
preds = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------
# 4. TRAIN MODEL
# -------------------------
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

# -------------------------
# 5. SAVE MODEL
# -------------------------
model.save("vegetable_classifier.h5")
print("Model Saved Successfully!")


ImportError: Traceback (most recent call last):
  File "c:\Users\SERVER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [35]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model("vegetable_classifier.h5")

img_path = r"C:\Users\victus\tomato_recognization\dataset\test2.jpg"   # your test image
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
class_idx = np.argmax(pred)
confidence = np.max(pred)

# Get class names
dataset_path = "dataset"
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator()
train_data = train_datagen.flow_from_directory(dataset_path)
class_names = list(train_data.class_indices.keys())

print("Prediction:", class_names[class_idx])
print("Confidence:", confidence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 949ms/step
Found 15334 images belonging to 15 classes.
Prediction: tomato
Confidence: 0.9969591
